# 🚀 Advanced RAG: Zero to Hero — A Guided Lab

Your RAG lab built the standard pipeline: retrieve, then generate. That works well for simple
questions, but breaks down on **ambiguous queries**, **questions needing multiple facts from
different documents**, and **queries phrased very differently from the source text**. This lab
teaches the techniques production RAG systems add on top: query rewriting, re-ranking, multi-hop
retrieval, and HyDE.

**Beginner-first.** Every chapter explains the *concept* before code. Prerequisite: the RAG lab
and the Embeddings & Search lab.

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ ASCII diagram → 🔬 Worked example →
⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. Where basic RAG breaks down
2. Query rewriting & expansion
3. Re-ranking with a cross-encoder (concept + implementation)
4. HyDE: Hypothetical Document Embeddings
5. Multi-hop retrieval (answering questions needing several facts)
6. Query routing (choosing the right retrieval strategy)
7. Contextual compression (trimming retrieved chunks)
8. Self-correction: checking your own retrieval
9. Combining everything: an advanced RAG pipeline
10. 🏆 Capstone: an advanced RAG assistant on a multi-hop question


In [ ]:
import numpy as np, re
from collections import Counter
print("Ready. We build on the basic RAG lab's retriever + MockLLM patterns.")

---
## Chapter 1 — Where Basic RAG Breaks Down

📖 **Theory.** Basic RAG (embed the query, retrieve top-k, generate) fails in predictable ways:
- **Vocabulary mismatch** — the query's wording doesn't match the document's wording, even
  though they mean the same thing.
- **Multi-hop questions** — the answer requires combining facts from **two or more separate**
  documents, but a single retrieval pass only grabs documents relevant to the *original* phrasing.
- **Noisy top-k** — the top-k chunks include some irrelevant ones that dilute the LLM's context
  or actively mislead it.

🖼️ **Diagram — where it breaks**
```
 "What's the warranty period?"     query mentions "warranty period"
        │
        ▼ retrieve                document says: "coverage lasts 12 months"  <- no shared words!
        │                          basic keyword/weak-semantic retrieval MISSES it
        ▼
   wrong/no answer
```

🧠 **Mental model.** Basic RAG is a single straight-line pipeline. Advanced RAG adds
**intelligence at each stage** — improving *what* gets searched for, *how* results get filtered,
and *whether* one retrieval pass is even enough.


In [ ]:
# A small knowledge base illustrating the vocabulary-mismatch problem
documents = [
    "Coverage lasts 12 months from the date of purchase for manufacturing defects.",
    "To request a repair, contact support with your order number.",
    "Our return window is 30 days from delivery with a valid receipt.",
    "Premium members get free expedited shipping on all orders.",
    "The mobile app requires iOS 15 or Android 10 or later.",
]

def tokenize(t): return set(re.findall(r"[a-z]+", t.lower()))

query = "what's the warranty period"
query_words = tokenize(query)
print("query words:", query_words)
for doc in documents:
    overlap = query_words & tokenize(doc)
    print(f"  overlap={len(overlap)}  '{doc[:50]}...'")
print("\nNotice: the warranty answer (doc 0, 'coverage lasts 12 months') shares almost NO words with the query!")

### ✏️ Your Turn 1.1
In a comment, name one real business question where the natural user phrasing is likely to
share very few words with the document that actually answers it.

In [ ]:
# your example


✅ **Solution**
```python
# "Can I get my money back?" vs a document saying "Refunds are processed within 5-7 days"
# -- "money back" and "refunds" mean the same thing but share zero words.
```

---
## Chapter 2 — Query Rewriting & Expansion

📖 **Theory.** Before retrieving, use an LLM to **rewrite** the query into better search terms,
or **expand** it into several phrasings — then retrieve using all of them and merge results.
This directly attacks the vocabulary-mismatch problem from Chapter 1.

🖼️ **Diagram — query expansion**
```
 original: "what's the warranty period?"
        │ LLM rewrites/expands
        ▼
 ["warranty period", "coverage duration", "how long is the warranty",
  "manufacturing defect coverage length"]
        │ retrieve with EACH, merge results
        ▼
 much better chance of matching "coverage lasts 12 months"
```


In [ ]:
class MockLLM:
    def generate(self, prompt):
        p = prompt.lower()
        if "rewrite" in p and "warranty" in p:
            return "warranty period | coverage duration | how long is the warranty | manufacturing defect coverage"
        if "hypothetical" in p and "warranty" in p:
            return "The warranty coverage lasts for a period of 12 months from the date of purchase, covering manufacturing defects."
        if "sub-question" in p or "decompose" in p:
            return "1. What is the return window?\n2. What is the warranty period?"
        return "mock response"

llm = MockLLM()

def rewrite_query(query, llm):
    prompt = f"Rewrite this query into 3-4 alternate search phrasings, separated by '|': {query}"
    response = llm.generate(prompt)
    return [q.strip() for q in response.split("|")]

expanded = rewrite_query("what's the warranty period", llm)
print("expanded queries:", expanded)

def multi_query_retrieve(queries, documents, k=2):
    all_hits = {}
    for q in queries:
        qw = tokenize(q)
        scored = sorted(documents, key=lambda d: -len(qw & tokenize(d)))[:k]
        for doc in scored:
            all_hits[doc] = all_hits.get(doc, 0) + 1   # count how many query variants retrieved it
    return sorted(all_hits.items(), key=lambda x: -x[1])

results = multi_query_retrieve(expanded, documents, k=2)
print("\nmerged results (doc, times retrieved across query variants):")
for doc, count in results[:3]:
    print(f"  [{count}x] {doc[:60]}...")

⚡ **Pro tip.** Documents retrieved by **multiple** query variants are strong signals of true
relevance — this simple "voting" is a cheap but effective re-ranking signal.

### ✏️ Your Turn 2.1
Write a query-expansion prompt (as a string, don't need to call the mock) for the query "how do
I cancel", aimed at generating alternate phrasings like "subscription cancellation" and
"stop billing".

In [ ]:
expansion_prompt = None
print(expansion_prompt)

✅ **Solution**
```python
expansion_prompt = "Rewrite this query into 3-4 alternate search phrasings, separated by '|': how do I cancel"
```

---
## Chapter 3 — Re-ranking with a Cross-Encoder

📖 **Theory.** Your embedding-based retriever (a **bi-encoder** — query and document embedded
*separately*, then compared) is fast but approximate. A **cross-encoder** re-ranker takes the
**query and a candidate document together** as one input and directly scores their relevance —
much more accurate, but too slow to run on the whole corpus. The standard pattern: retrieve a
wider set (e.g. top-20) cheaply with the bi-encoder, then **re-rank** just those 20 with the
expensive-but-accurate cross-encoder, keeping only the final top-k.

🖼️ **Diagram — retrieve wide, re-rank narrow**
```
 corpus (10,000 docs) ──bi-encoder, fast──► top-20 candidates
                                                    │
                                    cross-encoder, accurate (query+doc together)
                                                    │
                                              final top-3 to the LLM
```


In [ ]:
def bi_encoder_score(query, doc):
    # fast: independent embeddings, simple word overlap as our toy embedding proxy
    return len(tokenize(query) & tokenize(doc))

def cross_encoder_score(query, doc):
    """Simulates a cross-encoder: looks at query AND doc TOGETHER for a more nuanced score.
    Real cross-encoders are transformer models fine-tuned specifically for relevance scoring."""
    q_words, d_words = tokenize(query), tokenize(doc)
    overlap = len(q_words & d_words)
    # a cross-encoder can also weigh semantic closeness of concepts even without exact overlap
    concept_bonus = 0
    concept_pairs = [({"warranty","coverage"}, {"warranty","coverage","defects","months"}),
                      ({"cancel","cancellation"}, {"return","refund","subscription"})]
    for q_concept, d_concept in concept_pairs:
        if q_words & q_concept and d_words & d_concept:
            concept_bonus += 2
    return overlap + concept_bonus

query = "what's the warranty period"
# stage 1: bi-encoder retrieves a wide candidate set (here, all docs since our corpus is small)
candidates = documents

# stage 2: cross-encoder re-ranks
reranked = sorted(candidates, key=lambda d: -cross_encoder_score(query, d))
print("re-ranked results:")
for doc in reranked[:3]:
    print(f"  score={cross_encoder_score(query, doc)}  {doc[:60]}...")

⚠️ **Common trap.** Running a cross-encoder over your **entire** corpus defeats its purpose —
it's too slow at scale. Always use it as a **second stage** over a small candidate set from a
fast first-stage retriever, never as the primary retrieval method.

### ✏️ Your Turn 3.1
Compare the top result from `bi_encoder_score` alone vs. `cross_encoder_score` for the query
"how do I cancel". Does re-ranking change the top pick?

In [ ]:
query2 = "how do i cancel"
bi_top = None
cross_top = None
print(bi_top); print(cross_top)

✅ **Solution**
```python
bi_top = max(documents, key=lambda d: bi_encoder_score(query2, d))
cross_top = max(documents, key=lambda d: cross_encoder_score(query2, d))
# cross-encoder can surface the "return window" doc even with weak keyword overlap,
# thanks to the concept_bonus capturing "cancel" <-> "return/refund/subscription"
```

---
## Chapter 4 — HyDE: Hypothetical Document Embeddings

📖 **Theory.** **HyDE** flips the retrieval direction: instead of embedding the (short, sparse)
**query**, ask the LLM to first **write a hypothetical answer** to the query (even without real
knowledge — just a plausible-sounding one), then embed and search with **that** instead. A
generated answer often uses vocabulary much closer to the real documents than the original
short question does.

🖼️ **Diagram — HyDE's flip**
```
 STANDARD:  query "warranty period?" ──embed──► search (short, sparse query)

 HyDE:      query "warranty period?" ──LLM generates a plausible answer──►
            "The warranty typically covers defects for 12 months from purchase..."
                                              ──embed THIS──► search (richer, document-like text)
```


In [ ]:
def hyde_retrieve(query, documents, llm, k=2):
    prompt = f"Write a brief hypothetical answer to this question (it doesn't need to be accurate): {query}"
    hypothetical_doc = llm.generate(prompt)
    print("  hypothetical document generated:", hypothetical_doc[:80], "...")
    # now search using the HYPOTHETICAL document's words instead of the original short query
    hyp_words = tokenize(hypothetical_doc)
    scored = sorted(documents, key=lambda d: -len(hyp_words & tokenize(d)))
    return scored[:k]

print("Standard retrieval (using raw query words):")
standard = sorted(documents, key=lambda d: -len(tokenize("what's the warranty period") & tokenize(d)))[:2]
for d in standard: print(" ", d[:60], "...")

print("\nHyDE retrieval (using a generated hypothetical answer's words):")
hyde_results = hyde_retrieve("what's the warranty period", documents, llm, k=2)
for d in hyde_results: print(" ", d[:60], "...")

⚡ **Pro tip.** HyDE is especially effective for **short, sparse** queries (like single
keywords or terse questions) where there simply isn't enough vocabulary in the query itself to
match well against longer, richer documents.

### ✏️ Your Turn 4.1
In a comment, explain a risk of HyDE: what happens if the LLM's "hypothetical answer" is
confidently wrong or steers toward the wrong topic entirely?

In [ ]:
# your explanation


✅ **Solution**
```python
# If the hypothetical answer is wrong/off-topic, its embedding will point search toward
# the WRONG part of the document space -- HyDE can actively hurt retrieval if the LLM's
# "plausible guess" is misleading rather than merely imprecise. It works because the
# hypothetical doc's STYLE/VOCABULARY tends to match real docs, even if facts are wrong --
# but a badly wrong topic guess breaks that assumption.
```

---
## Chapter 5 — Multi-Hop Retrieval

📖 **Theory.** Some questions need **multiple, separate facts** combined — e.g., "Is a laptop
still under warranty if I bought it during the free-shipping promotion 13 months ago?" needs
both the **warranty period** *and* something about the **promotion timing**. **Multi-hop
retrieval**: decompose the question into sub-questions, retrieve for **each** separately, then
combine all retrieved context for generation.

🖼️ **Diagram — decompose, retrieve each, combine**
```
 "Is my item still under warranty AND can I still get a refund?"
        │ decompose
        ▼
 sub-Q1: "warranty period?"  ──retrieve──► doc about warranty
 sub-Q2: "refund window?"    ──retrieve──► doc about returns
        │
        ▼ combine all retrieved context
   generate answer using BOTH facts together
```


In [ ]:
def decompose_question(question, llm):
    prompt = f"Decompose this question into simple sub-questions, one per line: {question}"
    response = llm.generate(prompt)
    return [line.split(".",1)[-1].strip() for line in response.split("\n") if line.strip()]

def multi_hop_retrieve(question, documents, llm, k_per_hop=1):
    sub_questions = decompose_question(question, llm)
    print("sub-questions:", sub_questions)
    all_context = []
    for sq in sub_questions:
        qw = tokenize(sq)
        best = max(documents, key=lambda d: len(qw & tokenize(d)))
        all_context.append(best)
    return sub_questions, all_context

compound_question = "What is the return window and what is the warranty period?"
subqs, context = multi_hop_retrieve(compound_question, documents, llm)
print("\nretrieved context for each sub-question:")
for sq, doc in zip(subqs, context):
    print(f"  '{sq}' -> {doc[:55]}...")

⚠️ **Common trap.** Skipping decomposition and retrieving **once** for the whole compound
question often only surfaces documents relevant to *part* of the question — the retrieved
context ends up one-sided, and the LLM either guesses at the missing part or omits it silently.

### ✏️ Your Turn 5.1
Write your own 2-part compound question about this documents corpus (e.g. combining shipping and
app-requirements topics) and manually identify which two documents would need to be retrieved to
answer it fully.

In [ ]:
my_compound_question = None
needed_docs = None   # list the 2 relevant documents


✅ **Solution**
```python
my_compound_question = "Do premium members get faster shipping, and what OS does the app need?"
needed_docs = [documents[3], documents[4]]   # shipping doc + app-requirements doc
```

---
## Chapter 6 — Query Routing

📖 **Theory.** Not every query needs the same retrieval strategy. **Query routing** classifies
the incoming query first, then picks the best-suited path: a simple factual lookup might go
straight to standard retrieval; a compound question routes to multi-hop; a very short/vague
query routes to HyDE.

🖼️ **Diagram — routing to the right strategy**
```
                    ┌─► simple factual  ──► standard retrieval
 query ─► classify ─┼─► compound/multi-part ──► multi-hop retrieval
                    └─► short/vague     ──► HyDE
```


In [ ]:
def route_query(query):
    q = query.lower()
    if " and " in q or q.count("?") > 1:
        return "multi_hop"
    if len(query.split()) <= 3:
        return "hyde"
    return "standard"

test_queries = [
    "what is the return window and what is the warranty period",   # compound
    "warranty",                                                     # short/vague
    "what is your return policy for defective items",                # normal, specific
]
for q in test_queries:
    print(f"'{q}'  ->  route: {route_query(q)}")

### ✏️ Your Turn 6.1
Extend `route_query` so that any query containing the word "or" (like "warranty OR refund
policy?") also routes to `"multi_hop"` — since "or" often signals multiple distinct questions
being asked at once.

In [ ]:
def route_query_v2(query):
    pass
print(route_query_v2("warranty or refund policy?"))

✅ **Solution**
```python
def route_query_v2(query):
    q = query.lower()
    if " and " in q or " or " in q or q.count("?") > 1:
        return "multi_hop"
    if len(query.split()) <= 3:
        return "hyde"
    return "standard"
```

---
## Chapter 7 — Contextual Compression

📖 **Theory.** Retrieved chunks are often only **partially** relevant — a 200-word chunk might
have one sentence that matters and 190 words of noise. **Contextual compression** trims each
retrieved chunk down to just the relevant sentence(s) before handing it to the LLM, keeping the
prompt focused and shorter (cheaper, and less likely to distract the model).

🖼️ **Diagram — trim to what matters**
```
 retrieved chunk: "Our company was founded in 1995. We ship worldwide. Standard shipping
                   takes 3-5 business days. We also offer gift wrapping for a small fee."
                                          │  compress: keep only query-relevant sentences
                                          ▼
 compressed:      "Standard shipping takes 3-5 business days."
```


In [ ]:
def compress_context(query, chunk):
    query_words = tokenize(query)
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", chunk) if s.strip()]
    relevant = [s for s in sentences if tokenize(s) & query_words]
    return " ".join(relevant) if relevant else sentences[0]   # fallback: first sentence if no direct match

noisy_chunk = ("Our company was founded in 1995. We ship worldwide to over 40 countries. "
               "Standard shipping takes 3 to 5 business days. We also offer gift wrapping "
               "for a small additional fee at checkout.")
compressed = compress_context("how long does shipping take", noisy_chunk)
print("original chunk length:", len(noisy_chunk), "chars")
print("compressed chunk:", compressed)
print("compressed length:", len(compressed), "chars")

⚡ **Pro tip.** In production, compression is usually done with an LLM call ("extract only
the sentences relevant to this question from this passage") rather than simple keyword overlap
— but the keyword version here demonstrates the concept cheaply and quickly.

### ✏️ Your Turn 7.1
Compress `noisy_chunk` for the query `"gift wrapping fee"` and confirm the founding-year and
shipping-time sentences get dropped.

In [ ]:
compressed2 = None
print(compressed2)

✅ **Solution**
```python
compressed2 = compress_context("gift wrapping fee", noisy_chunk)
# should keep only the gift-wrapping sentence
```

---
## Chapter 8 — Self-Correction: Checking Your Own Retrieval

📖 **Theory.** Before generating a final answer, a self-correcting RAG system asks the LLM to
**grade** whether the retrieved context actually answers the question. If **not**, it can trigger
a fallback: rewrite the query and retrieve again, widen the search, or honestly decline.

🖼️ **Diagram — the self-check loop**
```
 retrieve ─► LLM: "does this context answer the question? yes/no"
                          │
                    yes ──┴──► generate final answer
                          │
                     no ──┴──► retry with a rewritten query (up to N attempts) or decline
```


In [ ]:
def grade_relevance(question, context, llm):
    q_words, c_words = tokenize(question), tokenize(context)
    overlap_ratio = len(q_words & c_words) / max(len(q_words), 1)
    return "relevant" if overlap_ratio >= 0.3 else "not_relevant"

def self_correcting_retrieve(question, documents, llm, max_retries=2):
    query = question
    for attempt in range(max_retries):
        candidates = sorted(documents, key=lambda d: -len(tokenize(query) & tokenize(d)))
        best = candidates[0]
        grade = grade_relevance(question, best, llm)
        print(f"  attempt {attempt}: query='{query[:40]}...' -> grade={grade}")
        if grade == "relevant":
            return best, attempt
        # not relevant -- rewrite and retry
        query = rewrite_query(question, llm)[0]   # take the first alternate phrasing
    return None, max_retries

result, attempts = self_correcting_retrieve("what's the warranty period", documents, llm)
print("\nfinal result:", result[:60] if result else "no relevant doc found", "...")

⚠️ **Common trap.** Without a `max_retries` cap, a self-correcting loop that never finds a
"relevant" grade could retry **forever**. Always bound retries and have a graceful fallback
("I don't know") ready — the same lesson from the Agents lab's loop-safety chapter.

### ✏️ Your Turn 8.1
Run `self_correcting_retrieve` for a query about something **not** in the corpus at all (e.g.
"do you accept cryptocurrency") and observe what happens when `max_retries` is exhausted.

In [ ]:
result2, attempts2 = self_correcting_retrieve("do you accept cryptocurrency", documents, llm)
print(result2, attempts2)

✅ **Solution**
```python
result2, attempts2 = self_correcting_retrieve("do you accept cryptocurrency", documents, llm, max_retries=2)
# result2 will likely be None after exhausting retries -- the system should then
# respond "I don't know" rather than force an answer from irrelevant context.
```

---
## 🏆 Chapter 9 — Combining Everything: an Advanced RAG Pipeline

Build an `AdvancedRAG` class that: **routes** the query, applies **query expansion** or **HyDE**
as appropriate, does **multi-hop** retrieval for compound questions, **compresses** retrieved
context, and **self-corrects** if the result looks irrelevant.

In [ ]:
class AdvancedRAG:
    def __init__(self, documents, llm):
        pass
    def answer(self, question):
        pass

# arag = AdvancedRAG(documents, llm)
# print(arag.answer("what is the return window and what is the warranty period"))


✅ **Solution**
```python
class AdvancedRAG:
    def __init__(self, documents, llm):
        self.documents = documents
        self.llm = llm

    def answer(self, question):
        route = route_query_v2(question)
        if route == "multi_hop":
            subqs, context_docs = multi_hop_retrieve(question, self.documents, self.llm)
            compressed = [compress_context(sq, doc) for sq, doc in zip(subqs, context_docs)]
            return {"route": route, "sub_questions": subqs, "context": compressed}
        elif route == "hyde":
            context_docs = hyde_retrieve(question, self.documents, self.llm, k=1)
            compressed = [compress_context(question, d) for d in context_docs]
            return {"route": route, "context": compressed}
        else:
            best, attempts = self_correcting_retrieve(question, self.documents, self.llm)
            compressed = [compress_context(question, best)] if best else []
            return {"route": route, "context": compressed, "self_correct_attempts": attempts}

arag = AdvancedRAG(documents, llm)
print(arag.answer("what is the return window and what is the warranty period"))
print(arag.answer("warranty"))
print(arag.answer("what is your return policy for defective items"))
```


---
## 🏆 Chapter 10 — Capstone: An Advanced RAG Assistant on a Multi-Hop Question

Put it all together on a realistic compound query, combining routing, multi-hop retrieval,
compression, and a final generated answer.

In [ ]:
def generate_final_answer(question, context_pieces, llm):
    context_str = " ".join(context_pieces)
    prompt = f"Context: {context_str}\nQuestion: {question}\nAnswer using only the context above."
    return llm.generate(prompt)

capstone_question = "What is the return window and what is the warranty period?"
print("Question:", capstone_question)

### ✏️ Capstone Tasks
1. Build an `AdvancedRAG` instance over `documents`.
2. Call `.answer()` on `capstone_question` and inspect the routing decision + retrieved context.
3. Feed the compressed context into `generate_final_answer` to produce the final response.
4. Print the full pipeline trace: route → sub-questions → context → final answer.

In [ ]:
# Your full capstone pipeline here


✅ **Capstone Solution**
```python
arag_final = AdvancedRAG(documents, llm)
result = arag_final.answer(capstone_question)
print("route:", result["route"])
print("sub-questions:", result.get("sub_questions"))
print("compressed context:", result["context"])

final_answer = generate_final_answer(capstone_question, result["context"], llm)
print("\\nFinal answer:", final_answer)
```

🎉 **You've mastered advanced RAG!** Query rewriting/expansion, cross-encoder re-ranking, HyDE,
multi-hop decomposition, query routing, contextual compression, and self-correcting retrieval —
the techniques that separate a demo RAG system from a production-grade one. Combine these with
the basic RAG pipeline (ingest/chunk/embed/index) from the earlier lab for a complete system.

---
### 📌 Concept Quick-Reference
**Query rewriting/expansion:** generate alternate phrasings to fight vocabulary mismatch
**Re-ranking:** fast bi-encoder retrieves wide, slow-but-accurate cross-encoder narrows
**HyDE:** embed a generated hypothetical answer instead of the raw (sparse) query
**Multi-hop:** decompose compound questions, retrieve per sub-question, combine context
**Query routing:** classify the query first, pick the retrieval strategy that fits
**Contextual compression:** trim retrieved chunks to just the relevant sentences
**Self-correction:** grade retrieved relevance; retry or decline rather than force a bad answer
**Combined:** route -> (expand/HyDE/multi-hop) -> retrieve -> compress -> self-check -> generate
